# 06 Pre-trained CNN Architectures (ResNet, VGG, Inception)

## 📚 Learning Objectives

By completing this notebook (~20 min), you will:
- Load **ResNet50**, **VGG16**, and **InceptionV3** from Keras Applications and inspect their structure
- Run a **forward pass** with one of them on a sample image and see output shape
- Understand why we use pre-trained architectures instead of designing from scratch

---

## 🌍 Real life

**Where is this used?** ResNet, VGG, Inception are used in **image search**, **medical imaging**, and **autonomous driving** as backbones for classification or detection.

**In this notebook we use** **pre-trained CNN architectures** (ResNet, VGG, Inception) to **extract features** from images. We use **pre-trained models** (instead of training from scratch) **because** they already learned useful features on ImageNet; we reuse them and save **data and time**.

**📌 Covers slide(s):** **20** — Transfer Learning (VGG, ResNet, fine-tuning). *Do this notebook after that slide.*

---

**Before starting:** Run the imports cell below. First run may download weights.

## Theory (short)

- **ResNet:** Residual connections (skip connections); very deep networks; avoids vanishing gradients.
- **VGG:** Stack of 3×3 convs; simple and deep; often used as a baseline.
- **Inception:** Multiple filter sizes in parallel (inception modules); efficient computation.
- **Pre-trained:** Trained on ImageNet (1.2M images, 1000 classes); we use them as **feature extractors** or **fine-tune** for our task.
- **We use pre-trained architectures** instead of training from scratch when we have limited data or similar domain (e.g. natural images).

## 📥 Inputs & 📤 Outputs

**Inputs:** TensorFlow/Keras, NumPy. We use a **random sample image** (or one from CIFAR/MNIST resized) so we don't need external files.

**Dataset:** Synthetic — random image (no download; used to show pretrained model input/output shape).

**Outputs:** Model summaries (layer count, params), output shape after forward pass, and a short comparison sentence.

## Step 1: Imports

In [1]:
import numpy as np

try:
    import tensorflow as tf
    from tensorflow.keras.applications import ResNet50, VGG16, InceptionV3
    HAS_TF = True
except Exception as e:
    err = str(e).lower()
    if "charset_normalizer" in err or "md__mypyc" in err or "partially initialized" in err:
        print("⚠️ Fix: pip install --upgrade charset-normalizer requests, then restart kernel.")
        raise RuntimeError("Fix: pip install --upgrade charset-normalizer requests, then restart kernel.") from e
    HAS_TF = False

print("TensorFlow:", "yes" if HAS_TF else "no")

TensorFlow: yes


## Step 2: Load ResNet50 (we use pre-trained ResNet instead of training from scratch to reuse ImageNet features)

In [2]:
if HAS_TF:
    resnet = ResNet50(weights="imagenet", include_top=True, input_shape=(224, 224, 3))
    resnet.trainable = False
    print("ResNet50 summary (first/last few layers):")
    resnet.summary()

ResNet50 summary (first/last few layers):


Model: "resnet50"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 25,636,712 (97.80 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 25,636,712 (97.80 MB)

## Step 3: Compare with VGG16 and InceptionV3 (same idea: pre-trained feature extractors)

In [3]:
if HAS_TF:
    vgg = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
    inc = InceptionV3(weights="imagenet", include_top=False, input_shape=(299, 299, 3))
    print("VGG16 params:", vgg.count_params())
    print("InceptionV3 params:", inc.count_params())
    print("ResNet50 params:", resnet.count_params())
    print("\nAll three are pre-trained on ImageNet; we use them as backbones for transfer learning.")

VGG16 params: 14714688
InceptionV3 params: 21802784
ResNet50 params: 25636712

All three are pre-trained on ImageNet; we use them as backbones for transfer learning.


## Step 4: Forward pass on a sample image (we use ResNet to get feature logits for one image)

In [4]:
if HAS_TF:
    sample = np.random.rand(1, 224, 224, 3).astype(np.float32)
    out = resnet(sample, training=False)
    print("Input shape:", sample.shape)
    print("Output shape (1000 ImageNet classes):", out.shape)
    print("Sample output (first 5 logits):", out.numpy().flatten()[:5])

Input shape: (1, 224, 224, 3)
Output shape (1000 ImageNet classes): (1, 1000)
Sample output (first 5 logits): [1.54980735e-04 3.07020964e-04 6.77680873e-05 1.16180585e-04
 6.34965691e-05]


## 🧩 Mini-exercise

**Try it:** In a new cell, load VGG16 and run a forward pass on the same sample image you used for ResNet. Compare the output shape and (if you printed it) the prediction. Or print the names of the last three layers of ResNet50.

---

## ✅ Summary

**What you did:** Loaded ResNet50, VGG16, InceptionV3; compared param counts; ran a forward pass with ResNet on a sample image.

**In real life you'd also:** Replace the top layer for your classes, freeze base and train head, or fine-tune last layers.

**The main idea:** Pre-trained CNNs (ResNet, VGG, Inception) give strong feature extractors; we use them instead of training from scratch to save data and time.

**Next:** `05_transfer_learning_cnns` freezes the base and trains a new head; `07_training_cnn_image_datasets` trains a CNN on CIFAR-10.